# Gurmar feature engineering

Builds the model table from `gurmar_panel_regular.parquet` (regular shelf prices, pack segments) with the logic in
`src/preprocessing/features.py`. Spec: `handoff.md` Sec. 3; decisions taken on 2026-10-06:

| Spec item | Implementation | Deviation / note |
|---|---|---|
| 3.1 lags 1/7/15/30 | calendar-day lags of the **causal regular price** `R` per segment | rows ≠ days (missing scrape days, outage) → daily grid first |
| 3.2 rolling 7/30, momentum | on `R`, shifted after the window | `min_periods` 4 / 15 so B1 keeps rows; trees handle NaN |
| 3.3 USD/TRY | Yahoo `TRY=X`, as of t−1, 1d / 7d / 30d change | — |
| + oil | Brent (`BZ=F`) × USD/TRY = Brent in TRY, as of t−1 | added on request; pump prices not included |
| 3.4 TUIK food index | monthly food CPI change, usable from the day after its release date (3rd of next month) | exact release lag instead of "shift 1 month" |
| + web index | paper Eq. 1–2 style: equal-weight mean regular log change per COICOP, 30-day sum, CPI-weighted store index | decision: both TUIK and web |
| 3.5 cyclic | dow / month / day-of-month sin-cos | + flags 1st / 15th / 16th / month end / holiday |
| 3.6 target | `target_pct_change = regular_t / R_{t−1} − 1` | base = causal regular price |
| sales | causal sale features (on sale, depth, days in sale) | ex-post sale flags never used as features; on sale days the target is the regular (pre-sale) price carried through the sale — dropping those rows would keep only permanent cuts among "on sale" rows (selection bias) |
| + | days since change, change counts, last change size, unit price (log, relative to category median), store / category repricing shares | — |

Target days are reported for two sets: **spec** (1st, 15th, last day, holidays) and **data** (1st, 14th–16th, holidays).

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # paths below are relative to the repository root
    os.chdir(Path.cwd().parent)

import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.preprocessing import features as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

PROCESSED = Path("data/processed")
EXTERNAL = Path("data/external")
config = json.loads((PROCESSED / "gurmar_blocks.json").read_text())
OUTAGE = tuple(config["outage"])
BLOCKS = config["blocks"]
SEED = 42

In [2]:
inputs = F.load_inputs(PROCESSED, EXTERNAL)
start = time.time()
feat = F.build_features(inputs, OUTAGE)
print(f"built in {time.time() - start:.0f}s: {len(feat):,} grid rows, {feat['segment_id'].nunique():,} segments, "
      f"{int(feat['usable_target'].sum()):,} usable target rows")
FEATURES = F.feature_columns()
print(f"{len(FEATURES)} features in {len(F.FEATURE_GROUPS)} groups (+ categorical {F.CATEGORICAL})")

2026-10-06 14:38:43,003 INFO src.preprocessing.features: grid: 1,155,981 rows, 7296 segments


2026-10-06 14:38:55,717 INFO src.preprocessing.features: usable target rows: 505,867


built in 44s: 1,155,981 grid rows, 7,296 segments, 505,867 usable target rows
50 features in 10 groups (+ categorical ['coicop', 'unit_basis'])


## Coverage per block

In [3]:
def block_rows(frame, block, part):
    lo, hi = (block["train_start"], block["train_end"]) if part == "train" else (block["test_start"], block["test_end"])
    return frame[frame["usable_target"] & frame["date"].between(lo, hi)]

rows = []
for b in BLOCKS:
    tr, te = block_rows(feat, b, "train"), block_rows(feat, b, "test")
    rows.append({
        "block": b["block"], "train_rows": len(tr), "test_rows": len(te),
        "test_spec_target_rows": int(te["is_target_spec"].sum()), "test_data_target_rows": int(te["is_target_data"].sum()),
        "test_change_rate_%": round(100 * (te["target_pct_change"].abs() > F.TOL).mean(), 2),
        "test_change_rate_spec_days_%": round(100 * (te.loc[te["is_target_spec"], "target_pct_change"].abs() > F.TOL).mean(), 2),
        "test_change_rate_data_days_%": round(100 * (te.loc[te["is_target_data"], "target_pct_change"].abs() > F.TOL).mean(), 2),
    })
pd.DataFrame(rows).set_index("block").T

block,B1,B2,B3
train_rows,189707.00,383008.00,445584.00
test_rows,139713.00,62576.00,60283.00
test_spec_target_rows,14063.00,7211.00,6967.00
test_data_target_rows,24010.00,14672.00,3247.00
test_change_rate_%,3.98,5.09,1.69
test_change_rate_spec_days_%,7.67,10.04,1.78
test_change_rate_data_days_%,5.64,8.25,2.09


In [4]:
usable = feat[feat["usable_target"]]
missing = pd.DataFrame({
    grp: [round(100 * usable[cols].isna().any(axis=1).mean(), 1),
          round(100 * usable[cols].isna().mean().mean(), 1)]
    for grp, cols in F.FEATURE_GROUPS.items()
}, index=["rows with any NaN %", "mean NaN share %"]).T
missing

,rows with any NaN %,mean NaN share %
lags,53.9,22.9
cyclic,0.0,0.0
usd_try,0.0,0.0
oil,0.0,0.0
tuik,0.0,0.0
rolling,71.7,16.1
sale,0.0,0.0
unit_price,9.7,9.7
cross_section,4.7,2.8
web_index,0.0,0.0


## Leakage test

For a cut-off day *T*, inputs are truncated after *T* and **all prices on day *T* are scrambled**.
If any feature for days ≤ *T* changes, it used information from day *T* or later. Run on a random subset of series
(the same subset in both runs, so cross-sectional features are comparable).

In [5]:
rng = np.random.default_rng(SEED)
subset = rng.choice(inputs["regular"]["series_id"].unique(), size=400, replace=False)

def subset_inputs(cutoff=None):
    out = dict(inputs)
    for key in ("regular", "mapped"):
        df = inputs[key][inputs[key]["series_id"].isin(subset)].copy()
        if cutoff is not None:
            df = df[df["date"] <= cutoff]
            on_cut = df["date"] == cutoff
            df.loc[on_cut, "price"] = df.loc[on_cut, "price"] * rng.uniform(0.3, 3.0, on_cut.sum())
        out[key] = df
    return out

logging.getLogger("src").setLevel(logging.WARNING)
reference = F.build_features(subset_inputs(), OUTAGE)

results = []
for cutoff in [pd.Timestamp("2026-04-15"), pd.Timestamp("2026-09-15")]:
    test = F.build_features(subset_inputs(cutoff), OUTAGE)
    keys = ["segment_id", "date"]
    a = reference[reference["date"] <= cutoff].set_index(keys)[FEATURES].sort_index()
    b = test[test["date"] <= cutoff].set_index(keys)[FEATURES].sort_index()
    a, b = a.align(b, join="inner")
    diff = (a - b).abs()
    both_nan = a.isna() & b.isna()
    changed = ((diff > 1e-9) | (a.isna() != b.isna())) & ~both_nan
    results.append(changed.sum().rename(str(cutoff.date())))

leak = pd.concat(results, axis=1)
print("features that changed:", leak[leak.sum(axis=1) > 0].index.tolist() or "none")
leak.sum().to_frame("changed cells (all features)")

features that changed: none


,changed cells (all features)
2026-04-15,0
2026-09-15,0


## Persistence baseline per block

Two persistence variants, both using only information up to t−1:
- **causal regular t−1** (`base_price`): treats a fresh price drop as a sale for up to 21 days, so it lags behind
  permanent price cuts;
- **shelf price t−1**: the last observed shelf price; right after a permanent cut, wrong the day after a sale ends.

Metrics follow handoff.md 5.1: MAE, RMSE, MAPE on price (TRY), directional accuracy (up / down / flat).
A model has to beat the **better** of the two.

In [6]:
def direction(x):
    return np.sign(np.where(np.abs(x) > F.TOL, x, 0))

def metrics(frame, pred):
    y = frame["target_price"].to_numpy()
    p = np.asarray(pred)
    err = p - y
    return {
        "rows": len(frame),
        "MAE": np.mean(np.abs(err)), "RMSE": np.sqrt(np.mean(err ** 2)),
        "MAPE_%": 100 * np.mean(np.abs(err) / y),
        "DA_%": 100 * np.mean(direction(frame["target_pct_change"]) == direction(p / frame["base_price"] - 1)),
        "changed_%": 100 * np.mean(np.abs(frame["target_pct_change"]) > F.TOL),
    }

rows = []
for b in BLOCKS:
    te = block_rows(feat, b, "test")
    for subset_name, mask in [("all days (global)", slice(None)), ("spec target days", te["is_target_spec"]),
                              ("data target days", te["is_target_data"])]:
        part = te[mask]
        shelf_pred = part["shelf_price_lag1"].fillna(part["base_price"])
        rows.append({"block": b["block"], "subset": subset_name, "baseline": "causal regular t-1",
                     **metrics(part, part["base_price"])})
        rows.append({"block": b["block"], "subset": subset_name, "baseline": "shelf price t-1",
                     **metrics(part, shelf_pred)})

baseline = pd.DataFrame(rows).set_index(["block", "subset", "baseline"]).round(4)
baseline.to_csv(PROCESSED / "baseline_persistence.csv")
baseline

rows      MAE     RMSE  MAPE_%     DA_%  changed_%
block subset            baseline                                                                
B1    all days (global) causal regular t-1  139713   1.8348  17.4635  1.1451  96.0168     3.9832
                        shelf price t-1     139713   7.0545  29.1164  3.5686  85.9140     3.9832
      spec target days  causal regular t-1   14063   3.7253  23.4183  2.2320  92.3274     7.6726
                        shelf price t-1      14063   9.0915  31.1240  4.9068  81.4335     7.6726
      data target days  causal regular t-1   24010   2.6747  20.6645  1.6034  94.3565     5.6435
                        shelf price t-1      24010   8.0826  29.4689  4.2698  83.2861     5.6435
B2    all days (global) causal regular t-1   62576   3.9757  43.0586  1.3362  94.9086     5.0914
                        shelf price t-1      62576  10.8285  47.1295  4.1283  84.1297     5.0914
      spec target days  causal regular t-1    7211   5.0447  37.3574  2.2029  89.9598    10.0402
                        shelf price t-1       7211   8.7719  39.7221  3.7359  84.2602    10.0402
      data target days  causal regular t-1   14672   4.9697  42.2287  1.9425  91.7530     8.2470
                        shelf price t-1      14672  10.1364  45.1055  4.0163  84.0376     8.2470
B3    all days (global) causal regular t-1   60283   1.1463  23.7219  0.3945  98.3096     1.6904
                        shelf price t-1      60283   4.5782  19.2498  2.6715  88.6983     1.6904
      spec target days  causal regular t-1    6967   0.7253   7.9475  0.3962  98.2202     1.7798
                        shelf price t-1       6967   4.7771  18.5972  2.8987  87.4982     1.7798
      data target days  causal regular t-1    3247   0.9274   9.2267  0.4857  97.9058     2.0942
                        shelf price t-1       3247   5.2642  19.6709  3.1794  86.3258     2.0942

## Save

In [7]:
keep = (["segment_id", "series_id", "date", "observed", "usable_target", "is_sale_day", "in_outage", "is_target_spec", "is_target_data",
         "shelf_price", "base_price", "target_price", "target_pct_change"] + F.CATEGORICAL + FEATURES)
feat[keep].to_parquet(PROCESSED / "gurmar_features.parquet", index=False)
(PROCESSED / "feature_groups.json").write_text(json.dumps(
    {"groups": F.FEATURE_GROUPS, "categorical": F.CATEGORICAL, "target": "target_pct_change", "base": "base_price"}, indent=2))
print(f"Saved {len(feat):,} rows × {len(keep)} columns → {PROCESSED / 'gurmar_features.parquet'}")

Saved 1,155,981 rows × 65 columns → data\processed\gurmar_features.parquet
